# Chapter Notes: Tensors and Image Classification

Notes from reading the fastai book, covering:
- Tensors: rank, shape, length, axes
- Indexing into tensors
- Image and video tensors
- Broadcasting
- Pixel-wise comparison for classification
- Loss vs metrics
- Underfitting vs overfitting

In [ ]:
import torch
from torch import tensor
import numpy as np

## Part 1: Tensor Fundamentals

### Terminology

| Term | Meaning | Example |
|------|---------|--------|
| **Scalar** | Single number | `5` |
| **Vector** | 1D array of numbers | `[1, 2, 3]` |
| **Matrix** | 2D grid of numbers | `[[1,2], [3,4]]` |
| **Tensor** | N-dimensional array (general term) | Any of the above |

### Rank, Shape, Length

- **Rank** = number of axes (dimensions)
- **Shape** = size of each axis
- **Length** = size of one specific axis

Note: "Dimension" is ambiguous - can mean rank OR length. Use specific terms.

In [ ]:
# Rank 0: Scalar (single value)
scalar = tensor(5)
print(f"Scalar: {scalar}")
print(f"Shape: {scalar.shape}")
print(f"Rank: {scalar.dim()}")
print()

In [ ]:
# Rank 1: Vector (1D array)
vector = tensor([1, 2, 3, 4, 5])
print(f"Vector: {vector}")
print(f"Shape: {vector.shape}")
print(f"Rank: {vector.dim()}")
print(f"Length: {vector.shape[0]}")
print()

In [ ]:
# Rank 2: Matrix (2D grid)
matrix = tensor([[1, 2, 3],
                 [4, 5, 6]])
print(f"Matrix:\n{matrix}")
print(f"Shape: {matrix.shape}")
print(f"Rank: {matrix.dim()}")
print(f"Rows (axis 0): {matrix.shape[0]}, Columns (axis 1): {matrix.shape[1]}")
print()

In [ ]:
# Rank 3: 3D tensor (e.g., batch of grayscale images)
images = torch.randn(10, 28, 28)  # 10 images, 28x28 pixels each
print(f"Batch of images shape: {images.shape}")
print(f"Rank: {images.dim()}")
print(f"10 images, each 28 rows x 28 columns")

## Part 2: Indexing Into Tensors

Each index you provide "drills down" one dimension.

Think: **How many indices do I need to get a single value?** That's the rank.

In [ ]:
# Create a batch of "images" with recognizable values
batch = torch.arange(24).reshape(2, 3, 4)  # 2 images, 3 rows, 4 cols
print(f"Full tensor shape: {batch.shape}")
print(f"Full tensor:\n{batch}\n")

In [ ]:
# One index: get first "image"
print(f"batch[0] shape: {batch[0].shape}")
print(f"batch[0]:\n{batch[0]}\n")

In [ ]:
# Two indices: get first row of first image
print(f"batch[0, 0] shape: {batch[0, 0].shape}")
print(f"batch[0, 0]: {batch[0, 0]}\n")

In [ ]:
# Three indices: get single value (one pixel)
print(f"batch[0, 0, 0] shape: {batch[0, 0, 0].shape}")
print(f"batch[0, 0, 0]: {batch[0, 0, 0]}")
print(f"This is a scalar - rank 0")

## Part 3: Image Tensors

### Channels

Each "channel" is one layer of information per pixel:
- Grayscale: 1 channel (intensity)
- RGB: 3 channels (red, green, blue)
- RGBA: 4 channels (RGB + transparency)

### Common Image Tensor Shapes

| Shape | Meaning |
|-------|--------|
| `(28, 28)` | One grayscale image |
| `(3, 224, 224)` | One RGB image |
| `(64, 1, 28, 28)` | Batch of 64 grayscale images |
| `(64, 3, 224, 224)` | Batch of 64 RGB images |

In [ ]:
# Simulate an RGB image
rgb_image = torch.randint(0, 256, (3, 8, 8))  # 3 channels, 8x8 pixels

print(f"RGB image shape: {rgb_image.shape}")
print(f"Red channel shape: {rgb_image[0].shape}")
print(f"Green channel shape: {rgb_image[1].shape}")
print(f"Blue channel shape: {rgb_image[2].shape}")

In [ ]:
# Get one pixel's RGB values (at row 4, col 4)
red = rgb_image[0, 4, 4]
green = rgb_image[1, 4, 4]
blue = rgb_image[2, 4, 4]

print(f"Pixel at (4,4): R={red}, G={green}, B={blue}")

## Part 4: Video Tensors (Rank 5)

Videos add a time dimension:

```
(batch, frames, channels, height, width)
```

In [ ]:
# Simulate a video batch
video_batch = torch.randn(8, 30, 3, 224, 224)

print(f"Video batch shape: {video_batch.shape}")
print(f"")
print(f"Breakdown:")
print(f"  Axis 0: {video_batch.shape[0]} videos")
print(f"  Axis 1: {video_batch.shape[1]} frames per video")
print(f"  Axis 2: {video_batch.shape[2]} color channels (RGB)")
print(f"  Axis 3: {video_batch.shape[3]} pixels height")
print(f"  Axis 4: {video_batch.shape[4]} pixels width")

In [ ]:
# Drill down through a video tensor
print(f"video_batch[0] shape: {video_batch[0].shape} <- first video")
print(f"video_batch[0, 0] shape: {video_batch[0, 0].shape} <- first frame")
print(f"video_batch[0, 0, 1] shape: {video_batch[0, 0, 1].shape} <- green channel")
print(f"video_batch[0, 0, 1, 100] shape: {video_batch[0, 0, 1, 100].shape} <- row 100")
print(f"video_batch[0, 0, 1, 100, 112] shape: {video_batch[0, 0, 1, 100, 112].shape} <- single pixel")

In [ ]:
# Memory calculation
total_values = 8 * 30 * 3 * 224 * 224
memory_bytes = total_values * 4  # float32 = 4 bytes
memory_gb = memory_bytes / (1024**3)

print(f"Total values: {total_values:,}")
print(f"Memory (float32): {memory_gb:.2f} GB")
print(f"This is why video models need serious GPU memory!")

## Part 5: Broadcasting

Broadcasting automatically "stretches" tensors to match shapes for operations.

**Key rule:** Dimensions of size 1 get repeated to match the other tensor.

No new memory allocated - it's a virtual expansion.

In [ ]:
# Simplest case: scalar broadcasts to all elements
a = tensor([1, 2, 3])
result = a + 10

print(f"{a} + 10 = {result}")
print(f"The 10 was 'broadcast' to [10, 10, 10]")

In [ ]:
# Practical example: normalize pixel values (0-255 to 0-1)
fake_image = tensor([[128, 255, 64],
                     [32, 192, 96]], dtype=torch.float32)

normalized = fake_image / 255

print(f"Original:\n{fake_image}\n")
print(f"Normalized (divided by 255):\n{normalized}")
print(f"\nThe 255 broadcast to match shape {fake_image.shape}")

In [ ]:
# Broadcasting across one dimension
matrix = tensor([[1, 2, 3, 4],
                 [5, 6, 7, 8],
                 [9, 10, 11, 12]])

row_to_add = tensor([10, 20, 30, 40])  # Shape (4,)

result = matrix + row_to_add

print(f"Matrix shape: {matrix.shape}")
print(f"Row shape: {row_to_add.shape}")
print(f"Result shape: {result.shape}")
print(f"\nMatrix:\n{matrix}")
print(f"\nRow to add: {row_to_add}")
print(f"\nResult (row added to EACH row of matrix):\n{result}")

In [ ]:
# What broadcasting is doing under the hood (conceptually)
print("Conceptually, the row gets 'stretched' to a matrix:")
print(f"\n[10, 20, 30, 40] becomes:")
expanded = row_to_add.unsqueeze(0).expand(3, -1)
print(f"{expanded}")
print(f"\nThen element-wise addition happens")

In [ ]:
# Broadcasting rules - what works and what doesn't
print("Broadcasting rules:")
print("- Compare shapes right-to-left")
print("- Dimensions match if: equal OR one of them is 1")
print("")
print("Works:")
print("  (3, 4) + (4,)   -> (3, 4)")
print("  (3, 4) + (1, 4) -> (3, 4)")
print("  (3, 1) + (1, 4) -> (3, 4)")
print("")
print("Fails:")
print("  (3, 4) + (2, 4) -> ERROR (3 != 2, neither is 1)")

## Part 6: Pixel-wise Comparison for Classification

The naive approach: compare each image to an "ideal" average image.

This is NOT how real neural networks work, but it demonstrates the concepts.

In [ ]:
# Simulate: 100 validation images of "3"s, each 28x28
valid_3s = torch.randn(100, 28, 28)

# Simulate: the "ideal" 3 (mean of training 3s)
mean_3 = torch.randn(28, 28)

print(f"Validation 3s shape: {valid_3s.shape}")
print(f"Mean 3 shape: {mean_3.shape}")

In [ ]:
# Step 1: Compute pixel-wise differences (broadcasting!)
differences = valid_3s - mean_3  # (100, 28, 28) - (28, 28)

print(f"Differences shape: {differences.shape}")
print(f"")
print(f"This is NOT the final answer!")
print(f"We have 100 images x 28 x 28 = {100 * 28 * 28:,} difference values")
print(f"We need to collapse this to 100 scores (one per image)")

In [ ]:
# Step 2: Take absolute value (we care about magnitude, not direction)
abs_differences = differences.abs()

print(f"Absolute differences shape: {abs_differences.shape}")

In [ ]:
# Step 3: Mean across pixels (collapse 28x28 to single number per image)
# dim=(1,2) means "take mean across axis 1 and axis 2 (the pixel dimensions)"
scores = abs_differences.mean(dim=(1, 2))

print(f"Scores shape: {scores.shape}")
print(f"")
print(f"Now we have 100 scores, one per image!")
print(f"Lower score = more similar to ideal 3")
print(f"")
print(f"First 10 scores: {scores[:10]}")

In [ ]:
# All in one line (L1 distance / Mean Absolute Error)
l1_scores = (valid_3s - mean_3).abs().mean(dim=(1, 2))

# Alternative: L2 distance (RMSE) - penalizes large errors more
l2_scores = ((valid_3s - mean_3) ** 2).mean(dim=(1, 2)).sqrt()

print(f"L1 (MAE) first 5: {l1_scores[:5]}")
print(f"L2 (RMSE) first 5: {l2_scores[:5]}")

### Limitations of Pixel-wise Comparison

This approach assumes:
- Every "3" is in the same position
- Every "3" is the same size  
- Every "3" is the same orientation
- One digit per image, centered

**It breaks with:**
- Shifted images ("3" in top-left corner)
- Rotated images
- Size variations
- Multiple digits in one image

**This is why we need neural networks** - they learn to detect features regardless of position.

## Part 7: Loss vs Metrics

| | Loss | Metric |
|-|------|--------|
| **For whom** | The model (training signal) | Humans (interpretability) |
| **Format** | Float, but abstract | Float, but meaningful |
| **Used for** | Adjusting weights via gradients | Deciding if model is good enough |
| **Requirements** | Must be differentiable | Just needs to measure what you care about |

The model only "sees" loss. Metrics are for you.

In [ ]:
# PyTorch provides loss functions
import torch.nn.functional as F

# Simulated predictions and targets
predictions = torch.randn(10)  # Model outputs
targets = torch.randn(10)      # Actual values

# L1 Loss (Mean Absolute Error)
l1_loss = F.l1_loss(predictions, targets)
print(f"L1 Loss: {l1_loss.item():.4f}")

# L2 Loss (Mean Squared Error)
mse_loss = F.mse_loss(predictions, targets)
print(f"MSE Loss: {mse_loss.item():.4f}")

In [ ]:
# Metric example: Accuracy (for classification)
# Simulated: 100 predictions, 100 actual labels
pred_labels = torch.randint(0, 10, (100,))  # Predicted digits 0-9
true_labels = torch.randint(0, 10, (100,))  # Actual digits 0-9

# How many did we get right?
correct = (pred_labels == true_labels).sum().item()
accuracy = correct / len(true_labels)

print(f"Correct: {correct}/100")
print(f"Accuracy: {accuracy:.1%}")
print(f"")
print(f"This is a METRIC - meaningful to humans")
print(f"The model doesn't use this directly for training")

## Part 8: Underfitting vs Overfitting

Track **training loss** and **validation loss** over time:

```
Loss
 │
 │ ╲
 │  ╲  Training loss (keeps dropping)
 │   ╲____________________________
 │    ╲
 │     ╲   Validation loss (drops, then...)
 │      ╲___________
 │                  ╲_____ starts rising = STOP
 │                         
 └──────────────────────────────────► Training time
      │           │
   Underfitting   Sweet spot   Overfitting
```

| State | Training Loss | Validation Loss | What's Happening |
|-------|---------------|-----------------|------------------|
| Underfitting | High | High | Model hasn't learned enough |
| Good fit | Low | Low | Model generalizes well |
| Overfitting | Very low | Rising | Model memorized training data |

In [ ]:
# Simulated training history
epochs = list(range(1, 21))

# Training loss keeps going down
train_loss = [1.0 / (0.5 + 0.1 * e) for e in epochs]

# Validation loss goes down, then up (overfitting!)
val_loss = [1.0 / (0.5 + 0.1 * e) + 0.02 * max(0, e - 10) ** 1.5 for e in epochs]

print("Epoch | Train Loss | Val Loss | Status")
print("-" * 45)
for e, tl, vl in zip(epochs, train_loss, val_loss):
    if e <= 5:
        status = "Underfitting"
    elif e <= 12:
        status = "Good fit"
    else:
        status = "Overfitting!"
    print(f"  {e:2d}  |   {tl:.4f}   |  {vl:.4f}  | {status}")

## Summary

### Key Concepts

1. **Tensors** are N-dimensional arrays. Rank = number of axes, shape = size of each axis.

2. **Indexing** drills down dimensions. Full indices → single value.

3. **Image tensors** use channels for color. Common shape: `(batch, channels, height, width)`.

4. **Broadcasting** auto-expands tensors for operations. Dimensions of 1 stretch to match.

5. **Pixel-wise comparison** is naive but demonstrates concepts. Neural networks are needed for real robustness.

6. **Loss** is for training (model uses it). **Metrics** are for humans (we interpret them).

7. **Overfitting** = model memorizes training data. Watch validation loss to detect it.

### Next: Stochastic Gradient Descent (SGD)

How models actually learn by adjusting weights to minimize loss.